# Exercise 04 — consumer groups & manual offsets

**Goal:** understand *how Kafka tracks what you've already processed*, what happens when a consumer crashes mid-message, and how to read old events again (replay). Along the way you create a topic yourself with `rpk`.

**Prerequisite:** there is data in `rider-locations` (run [`exercise_01_produce_single.ipynb`](exercise_01_produce_single.ipynb) or [`exercise_03_produce_batch.ipynb`](exercise_03_produce_batch.ipynb)).

## Background — auto-commit vs manual commit

By default the consumer **auto-commits** its offset every 5 seconds. That's convenient — but risky:

```
1. consumer.poll() returns message X
2. you process it (write the rider position to the database, call an API, …)
3. ❗ your process CRASHES before the next auto-commit fires
4. on restart the consumer asks: 'where were we?' → broker says X
5. message X is processed AGAIN → duplicate work
```

Or worse, the *opposite* failure: with auto-commit on, the offset can advance *before* you finish processing → on crash the message is **lost** entirely (At-Most-Once).

With `enable.auto.commit: False` **you** decide when to commit. The safe pattern is:

```
read message → process (must be idempotent or transactional) → commit
```

If processing fails the offset is never committed → the message is re-delivered → **At-Least-Once**. Combined with idempotent processing (e.g. `INSERT … ON CONFLICT DO NOTHING`), that gives you effectively exactly-once semantics.

## Step 1 — read with manual commit

Run the cell. Each iteration prints `read — processing… committed.` → that's the safe pattern in action.

In [ ]:
from confluent_kafka import Consumer

BROKER = 'redpanda:29092'

consumer = Consumer({
    'bootstrap.servers':  BROKER,
    'group.id':           'manual-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,           # we drive commits ourselves
})
consumer.subscribe(['rider-locations'])

print(f'{"#":>3} | {"Key":>7} | {"Offset":>6} | Status')
print('-' * 45)

messages_read, empty_polls = 0, 0
while messages_read < 5 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    key = msg.key().decode() if msg.key() else 'None'
    print(f'{messages_read:>3} | {key:>7} | {msg.offset():>6} | '
          f'read — processing...', end='')
    # Imagine writing the rider position to the live-map database here.
    # If that raises, we never reach commit().
    consumer.commit(message=msg)   # advance the offset
    print(' committed.')

consumer.close()
print(f'\nProcessed and committed {messages_read} messages.')

## Task A — simulate a crash by skipping the commit

Modify the cell above so that `consumer.commit(message=msg)` is *not* called (comment it out). Run twice **with the same `group.id`** (a fresh one such as `'no-commit-demo'` makes the effect easiest to see).

**Expected:** the second run shows you the *same* messages with the same offsets — Kafka thinks you've never processed them. Real-world consequence: you'd write the same rider positions to the database twice. That's why all downstream sinks should be **idempotent** when using At-Least-Once.

In [ ]:
# TODO: copy the manual-commit cell above and remove the
#       consumer.commit(...) line. Run twice with the same group.id.

## Task B — two consumers, one group

Here's the magic of consumer groups: **each partition is owned by exactly one consumer in the group**. Our topics have 3 partitions each. With 1 consumer, that one gets everything. With 2 consumers, the work is split: one owns two partitions of each topic, the other one. With 4 consumers, one of them sits idle.

**To experiment:**

1. Right-click this notebook in the Explorer → *Open in New Tab* → in *both* tabs run the cell below.
2. In a third tab, run [`exercise_03_produce_batch.ipynb`](exercise_03_produce_batch.ipynb).
3. **Watch:** which tab gets which messages? Why? Look at the partition numbers.

The cell stops by itself after 5 minutes, or earlier when you press ■.

In [ ]:
from confluent_kafka import Consumer
from datetime import datetime
import time

BROKER = 'redpanda:29092'
LISTEN_SECONDS = 300   # stops by itself after 5 minutes; stop earlier with ■

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'group-experiment',
    'auto.offset.reset': 'latest',         # only NEW events from now
})
consumer.subscribe(['rider-locations', 'orders'])
print('Listening — start the producer in another tab. Stop with the ■ button.')

deadline = time.time() + LISTEN_SECONDS
try:
    while time.time() < deadline:
        msg = consumer.poll(0.5)
        if msg is None or msg.error(): continue
        ts = datetime.now().strftime('%H:%M:%S')
        key = msg.key().decode() if msg.key() else 'None'
        print(f'[{ts}] {msg.topic()} P{msg.partition()} '
              f'offset={msg.offset()} key={key}')
    print('Time is up.')
except KeyboardInterrupt:
    print('\nStopped.')
finally:
    consumer.close()

## Task C — create a topic yourself

So far the smoke test created the topics for you. In a real project you create topics deliberately: how many **partitions** (how many consumers of a group can work in parallel), how many **replicas** (copies on other brokers), and how long the broker keeps the events (**retention**).

Open a terminal (*Terminal → New Terminal*) and create the topic `order-replay` with 3 partitions, 1 replica and a retention of **1 day**:

```bash
rpk topic create order-replay --partitions 3 --replicas 1 \
    --topic-config retention.ms=TODO \
    -X brokers=redpanda:29092
```

`retention.ms` is in milliseconds — compute one day. Then look at your topic:

```bash
rpk topic describe order-replay -X brokers=redpanda:29092
rpk topic describe order-replay -p -X brokers=redpanda:29092
```

**Questions**

- Where do you find your `retention.ms` in the output, and what does the column `SOURCE` say about it? Compare with `retention.bytes`.
- Why `--replicas 1` here? What would you choose in a cluster with three brokers?
- What happens to an order event after one day?

Then run the cell below **once**: it writes 10 orders into `order-replay` and remembers a moment `MIDDLE` between the first and the last five.

In [ ]:
from confluent_kafka import Producer
import json, time, random

BROKER = 'redpanda:29092'
TOPIC  = 'order-replay'

producer  = Producer({'bootstrap.servers': BROKER})
customers = ['Anna', 'Ben', 'Clara', 'David', 'Elif']


def send_orders(first, last):
    """Send the orders order-<first> ... order-<last> with status 'placed'."""
    for n in range(first, last + 1):
        order_id = f'order-{n}'
        event = {
            'order_id':   order_id,
            'customer':   random.choice(customers),
            'restaurant': 'Pizzeria Roma',
            'total_chf':  round(random.uniform(15, 50), 2),
            'status':     'placed',
            'timestamp':  time.time(),
        }
        producer.produce(TOPIC, key=order_id, value=json.dumps(event))
    producer.flush()


send_orders(5001, 5005)
time.sleep(3)
MIDDLE = int(time.time())   # remember this moment (UNIX seconds) for the replay by time
time.sleep(3)
send_orders(5006, 5010)
print(f'Sent 10 orders. order-5006 ... order-5010 came after MIDDLE = {MIDDLE}')

## Task D — replay: read old events again

A consumer group remembers its position (the committed offset). Usually that's exactly what you want: after a restart you continue where you stopped. Sometimes you want to read **old events again** — for example after fixing a bug in your processing code, or to fill a new database. Kafka keeps the events until the retention runs out, so you only move the group's position back. That's a **replay**.

### Step 1 — read everything once

Run the cell below. It reads all orders of `order-replay` with the group `replay-demo` and commits its position when it closes. Then run it a **second time**: it reads nothing, the group is already at the end.

In [ ]:
from confluent_kafka import Consumer


def read_orders(group_id='replay-demo'):
    """Read everything the group has not seen yet, then close (commits the position)."""
    consumer = Consumer({
        'bootstrap.servers': BROKER,
        'group.id':          group_id,
        'auto.offset.reset': 'earliest',   # a NEW group starts at the beginning
    })
    consumer.subscribe([TOPIC])
    read, empty_polls = 0, 0
    try:
        while empty_polls < 5:
            msg = consumer.poll(2.0)
            if msg is None: empty_polls += 1; continue
            if msg.error(): continue
            empty_polls = 0; read += 1
            print(f'P{msg.partition()} offset={msg.offset()} key={msg.key().decode()}')
    finally:
        consumer.close()   # commits the last position and leaves the group
    print(f'{read} orders read with group {group_id!r}')


read_orders()

### Step 2 — look at the group and move it back

In the terminal:

```bash
rpk group describe replay-demo -X brokers=redpanda:29092
```

Look at `CURRENT-OFFSET`, `LOG-END-OFFSET` and `LAG` per partition, and at `STATE`.

Now move the group back to the start of the topic. Find the right command with `rpk group seek --help`:

```bash
# TODO: rpk group seek ... -X brokers=redpanda:29092
```

Run `rpk group describe` again — what changed? Then run the cell below.

In [ ]:
read_orders()   # same group as before -- what do you get now?

### Step 3 — replay from a point in time

`rpk group seek` also accepts a UNIX timestamp instead of `start`. Print `MIDDLE`, move the group to that moment, and read again. Which orders come back?

In [ ]:
print(MIDDLE)
# TODO: in the terminal: rpk group seek replay-demo --to <MIDDLE> -X brokers=redpanda:29092
#       then run the next line
read_orders()

### Step 4 — why the group must be inactive

Run the cell below: it keeps a consumer in the group `replay-demo` for 60 seconds. **While it runs**, try `rpk group seek replay-demo --to start -X brokers=redpanda:29092` in the terminal and read the `ERROR` column. Also look at `STATE` and `MEMBERS` in `rpk group describe`.

In [ ]:
import time

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'replay-demo',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe([TOPIC])
print('Member of the group replay-demo for 60 s -- run the seek in the terminal now.')
deadline = time.time() + 60
try:
    while time.time() < deadline:
        consumer.poll(0.5)
finally:
    consumer.close()
print('Left the group.')

**Questions**

- What does the seek report while the consumer is running? Why does the broker refuse it?
- In production your consumer runs as a service around the clock. In which order do you do a replay?

### Step 5 — optional: replay from Python

Instead of moving the group with `rpk`, a consumer can choose its start position itself. `offsets_for_times()` asks the broker for the first offset at or after a timestamp, per partition; `assign()` starts reading exactly there. Complete the cell.

In [ ]:
from confluent_kafka import Consumer, TopicPartition

consumer = Consumer({
    'bootstrap.servers':  BROKER,
    'group.id':           'replay-python',
    'enable.auto.commit': False,
})

# One entry per partition; the "offset" field carries the timestamp in MILLISECONDS.
wanted = [TopicPartition(TOPIC, p, MIDDLE * 1000) for p in range(3)]

# TODO: ask the broker for the first offset at or after MIDDLE in every partition
#       (consumer.offsets_for_times(...)) and start reading there (consumer.assign(...)).
offsets = ...

read, empty_polls = 0, 0
while empty_polls < 3:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; read += 1
    print(f'P{msg.partition()} offset={msg.offset()} key={msg.key().decode()}')
consumer.close()
print(f'{read} orders read from MIDDLE on')

## What you learned

- Auto-commit is convenient but trades correctness for simplicity.
- **At-Least-Once** = manual commit *after* processing → duplicates on retry, no data loss.
- **At-Most-Once** = commit *before* processing → no duplicates, possible loss.
- A consumer group **partitions the work** by partition. Adding more consumers than partitions does *not* speed things up — the extras sit idle.
- Topics are created deliberately: partitions (parallelism), replicas (copies on other brokers), retention (how long events are kept — and how far back a replay can go).
- **Replay** = move a group's committed offset back (`rpk group seek --to start` or `--to <timestamp>`) and read the old events again. Only possible while the group has no active members.
- `offsets_for_times()` + `assign()` lets a single consumer start at a point in time without touching any group.